In [1]:
import pandas as pd
import numpy as np
 
# =========================================================
# 0. CONFIGURATION & LOAD
# =========================================================
MATRIX_IN = "ml_cohort_master_v2.csv"
MATRIX_OUT = "ml_cohort_cleaned_v3.csv"
 
print("Loading Master Matrix...")
df = pd.read_csv(MATRIX_IN)
initial_rows, initial_cols = df.shape

# Separate metadata from actual physiological features
meta_cols = [
    "subject_id", "index_hadm_id", "index_time", "is_aortic_dissection", 
    "admission_location", "encounter_urgency", "gender", "race", 
    "insurance", "marital_status", "anchor_year_group"
]
features = [col for col in df.columns if col not in meta_cols ]
# =========================================================
# 1. ROW FILTERING (Drop patients missing >50% of features)
# =========================================================
print("\n--- Step 1: Patient (Row) Filtering ---")
# Calculate the percentage of missing features per patient
row_missing_pct = df[features].isnull().mean(axis=1)

# Keep patients with <= 50% missing data
df_rows_cleaned = df[row_missing_pct <= 0.50].copy()
dropped_rows = initial_rows - len(df_rows_cleaned)

print(f"Dropped {dropped_rows:,} patients who were missing >50% of their data.")
print(f"Remaining Patients: {len(df_rows_cleaned):,}")
 
# =========================================================
# 2. COLUMN FILTERING (Drop features missing in >50% of remaining patients)
# =========================================================
print("\n--- Step 2: Feature (Column) Filtering ---")
# Calculate missingness per feature on the newly cleaned patient pool
col_missing_pct = df_rows_cleaned[features].isnull().mean(axis=0)

# Identify features to keep
keep_features = col_missing_pct[col_missing_pct <= 0.50].index.tolist()
dropped_features = len(features) - len(keep_features)

df_cleaned = df_rows_cleaned[meta_cols + keep_features].copy()

print(f"Dropped {dropped_features:,} features missing in >50% of patients.")
print(f"Remaining Features: {len(keep_features):,}")

# Export the cleaned matrix
df_cleaned.to_csv(MATRIX_OUT, index=False)

 
# =========================================================
# 3. THE LEAKAGE AUDIT (Special Labs)
# =========================================================

print("\n--- Step 3: Test-Ordering Leakage Audit ---")
# We will check the '_mean' column for each special lab to see if it was ordered at all
special_labs = ['ddimer', 'troponin_t', 'troponin_i', 'crp', 'fibrinogen', 'inr']

# Create a dataframe to hold our audit results
audit_records = []

for lab in special_labs:
    target_col = f"{lab}_mean"
    # Check if the lab survived the 50% column drop
    survived = "Yes" if target_col in df_cleaned.columns else "No (Dropped in Step 2)"
    # Calculate missingness by cohort using the pre-column-drop dataframe (df_rows_cleaned)
    # so we can see the true distribution even if it was dropped
    if target_col in df_rows_cleaned.columns:
        target_missing = df_rows_cleaned[df_rows_cleaned['is_aortic_dissection'] == 1][target_col].isnull().mean()
        control_missing = df_rows_cleaned[df_rows_cleaned['is_aortic_dissection'] == 0][target_col].isnull().mean()
    else:
        target_missing, control_missing = np.nan, np.nan
    audit_records.append({
        "Lab Concept": lab.upper(),
        "Survived 50% Cut?": survived,
        "Target Null %": f"{target_missing*100:.1f}%",
        "Control Null %": f"{control_missing*100:.1f}%",
        "Absolute Discrepancy": f"{abs(control_missing - target_missing)*100:.1f}%"
    })

audit_df = pd.DataFrame(audit_records)
print(audit_df.to_string(index=False))
 

Loading Master Matrix...

--- Step 1: Patient (Row) Filtering ---
Dropped 312,415 patients who were missing >50% of their data.
Remaining Patients: 52,211

--- Step 2: Feature (Column) Filtering ---
Dropped 54 features missing in >50% of patients.
Remaining Features: 124

--- Step 3: Test-Ordering Leakage Audit ---
Lab Concept      Survived 50% Cut? Target Null % Control Null % Absolute Discrepancy
     DDIMER No (Dropped in Step 2)         99.3%          99.2%                 0.2%
 TROPONIN_T No (Dropped in Step 2)         89.4%          82.2%                 7.2%
 TROPONIN_I No (Dropped in Step 2)          nan%           nan%                 nan%
        CRP No (Dropped in Step 2)         99.3%          95.0%                 4.4%
 FIBRINOGEN No (Dropped in Step 2)         50.0%          83.5%                33.5%
        INR                    Yes          5.5%          19.8%                14.3%


In [2]:
import pandas as pd

df = pd.read_csv("ml_cohort_cleaned_v3.csv")

remaining_aa_patients = df[df['is_aortic_dissection'] == 1].shape[0]
print(f"\nRemaining Aortic Dissection Patients: {remaining_aa_patients:,}")


Remaining Aortic Dissection Patients: 454
